In [ ]:
from pyspark.sql.types import *

schema = StructType([
    StructField("YEAR_MONTH", StringType(), True),

    StructField("ICB_NAME", StringType(), True),
    StructField("ICB_CODE", StringType(), True),

    StructField("PRACTICE_NAME", StringType(), True),
    StructField("PRACTICE_CODE", StringType(), True),

    StructField("BNF_CHEMICAL_SUBSTANCE", StringType(), True),
    StructField("BNF_PRESENTATION_NAME", StringType(), True),
    StructField("BNF_CHAPTER_PLUS_CODE", StringType(), True),

    StructField("QUANTITY", IntegerType(), True),
    StructField("ITEMS", IntegerType(), True),

    StructField("ACTUAL_COST", DecimalType(18, 2), True)
])

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pathlib import Path

# ----------------------------------------------------
# 1. Spark session (Codespaces-optimised)
# ----------------------------------------------------
spark = SparkSession.builder \
    .appName("NHS_CSV_to_Parquet") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.default.parallelism", "8") \
    .config("spark.sql.files.maxPartitionBytes", 64 * 1024 * 1024) \
    .getOrCreate()

# ----------------------------------------------------
# 2. Schema
# ----------------------------------------------------
schema = StructType([
    StructField("YEAR_MONTH", StringType(), True),

    StructField("ICB_NAME", StringType(), True),
    StructField("ICB_CODE", StringType(), True),

    StructField("PRACTICE_NAME", StringType(), True),
    StructField("PRACTICE_CODE", StringType(), True),

    StructField("BNF_CHEMICAL_SUBSTANCE", StringType(), True),
    StructField("BNF_PRESENTATION_NAME", StringType(), True),
    StructField("BNF_CHAPTER_PLUS_CODE", StringType(), True),

    StructField("QUANTITY", IntegerType(), True),
    StructField("ITEMS", IntegerType(), True),

    StructField("ACTUAL_COST", DecimalType(18, 2), True)
])

# ----------------------------------------------------
# 3. Paths (Codespaces-safe)
# ----------------------------------------------------
BASE_DIR = Path("/workspaces/spark-test")

input_path = BASE_DIR / "spark_input"
output_path = BASE_DIR / "spark_output"

# If there is only ONE CSV file in spark_input, we auto-pick it
csv_files = list(input_path.glob("*.csv"))

if len(csv_files) == 0:
    raise FileNotFoundError("No CSV files found in spark_input folder")

input_file = str(csv_files[0])

print(f"Reading: {input_file}")
print(f"Writing to: {output_path}")

# ----------------------------------------------------
# 4. Read CSV
# ----------------------------------------------------
df = spark.read \
    .option("header", True) \
    .option("mode", "PERMISSIVE") \
    .schema(schema) \
    .csv(input_file)

# ----------------------------------------------------
# 5. Write Parquet
# ----------------------------------------------------
df.write \
    .mode("overwrite") \
    .parquet(str(output_path))

print("Done: CSV successfully converted to Parquet")

Reading: /workspaces/spark-test/spark_input/test.csv
Writing to: /workspaces/spark-test/spark_output


26/06/27 15:54:26 WARN CSVHeaderChecker: Number of column in CSV header is not equal to number of fields in the schema:
 Header length: 27, schema size: 11
CSV file: file:///workspaces/spark-test/spark_input/test.csv


Done: CSV successfully converted to Parquet


In [4]:


import urllib.request

url = "https://opendata.nhsbsa.net/dataset/906115a6-4155-44be-8b81-f8e83cebfb84/resource/6b52baa3-c550-4b1d-8cd2-46d40d840353/download/epd_snomed_202603.csv"
urllib.request.urlretrieve(url, "/workspaces/spark-test/spark_input/epd_snomed_202603.csv")


('/workspaces/spark-test/spark_input/epd_snomed_202603.csv',
 <http.client.HTTPMessage at 0x7ff5dc08a290>)